# Pré-processamento

A clusterização agrupa municípios semelhantes, e a semelhança se traduz, na
prática, em uma distância entre duas listas de 22 números. O cálculo só é
válido se as 22 colunas estiverem em condições comparáveis, o que não ocorre:
algumas variam de 0 a 950, outras de 0 a 100 e outras de 1 a 5, de modo que a
coluna de maior escala determinaria quase sozinha o agrupamento.

Este notebook estabelece essas condições em quatro passos e gera a
**Figura 3** do artigo:

1. **`log1p`** nas variáveis de cauda longa, de maneira que os municípios com
   valores extremos não dominem o cálculo (decidido no notebook 02);
2. **padronização**, de modo a uniformizar a escala das colunas;
3. **peso por bloco**, de modo a equalizar a contribuição das três fontes de
   dados;
4. **PCA**, empregado aqui apenas como diagnóstico da dimensionalidade
   efetiva dos dados.

A saída é o `matriz_modelagem.csv`, insumo da clusterização na próxima
entrega. Nenhum agrupamento é executado aqui.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

# Os notebooks ficam em notebooks/ e o código do projeto em src/. Estas duas
# linhas apontam o Python para src/, para os "from config import ..." abaixo
# funcionarem tanto rodando daqui quanto da raiz do projeto.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

from config import BASE_FINAL_CSV, DICIONARIO_CSV, DATA_PROCESSED
from estilo import aplicar_estilo, salvar
from figuras import plot_orcamento_blocos, plot_variancia_pca, plot_pc1_pc2

aplicar_estilo()   # mesma fonte, cores e grade em todas as figuras do artigo
# Se algum import acima falhar, quase sempre é o editor apontando para outra
# instalação do Python. O caminho impresso aqui é o que precisa ter as
# bibliotecas do requirements.txt.
print("Python:", sys.executable)

# O código do IBGE é identificador, não quantidade: lido como texto para não
# virar número em nenhuma etapa.
base = pd.read_csv(BASE_FINAL_CSV, dtype={"codigo_ibge": str})
dic = pd.read_csv(DICIONARIO_CSV)

## 1. Features utilizadas

A lista de features vem do dicionário, como no notebook 02, e a capital fica
de fora por não ter IEGM, condição indicada pela coluna `flag_sem_iegm`.
Restam 644 municípios e 22 features: 9 de criminalidade, 6 socioeconômicas e 7
de gestão.

A lista `log_cols` delimita as colunas que recebem `log1p`: as 9 taxas
criminais e as 2 variáveis monetárias, identificadas como de cauda longa no
notebook 02. Percentuais e notas de 1 a 5 dispensam a transformação, por
ocuparem faixa curta e sem valores extremos.

In [ ]:
bloco_de = dic.set_index("coluna")["bloco"]
ORDEM_BLOCOS = ["criminalidade", "socioeconomico", "gestao"]
# Mesma ordenação por bloco do notebook 02: mantém as colunas na mesma ordem
# em todas as tabelas e figuras do projeto.
features = sorted(dic.loc[dic["papel"] == "feature", "coluna"],
                  key=lambda c: (ORDEM_BLOCOS.index(bloco_de[c]), c))

# reset_index porque tirar a capital abre um buraco na numeração das linhas,
# e mais para baixo as posições precisam bater com as da matriz do sklearn.
modelagem = base[~base["flag_sem_iegm"]].reset_index(drop=True)

# As duas listas dividem as colunas em "recebe log1p" e "não recebe".
log_cols = ([c for c in features if bloco_de[c] == "criminalidade"]
            + ["pib_percapita", "renda_domiciliar_mediana"])
demais_cols = [c for c in features if c not in log_cols]
print(f"{len(modelagem)} municípios | {len(features)} features | "
      f"log1p em {len(log_cols)}")

## 2. Transformação e padronização

**Padronizar** uma coluna consiste em subtrair a sua média e dividir pelo
desvio padrão, de modo que a coluna passe a ter média 0 e desvio 1 e cada
valor represente desvios em relação à média, e não reais, percentuais ou
ocorrências por 100 mil habitantes. É o que torna somáveis, em uma mesma
distância, 22 colunas de unidades distintas.

O `ColumnTransformer` é o recurso do `scikit-learn` para aplicar tratamentos
distintos a grupos de colunas em uma única passagem: aqui, `log1p` seguido de
padronização nas colunas de cauda longa e apenas padronização nas demais. A
formulação deixa o objeto pronto para reaplicação idêntica na clusterização.

Detalhe relevante: a média e o desvio de cada coluna são calculados **sobre os
644 municípios que serão clusterizados**, e não sobre a base completa. A
inclusão da capital alteraria a escala de todos os municípios sem que ela
participasse do agrupamento.

In [ ]:
# Cada tupla do ColumnTransformer é (nome, o que fazer, em quais colunas).
pre = ColumnTransformer([
    # Cauda longa: primeiro log1p, depois padroniza (Pipeline = em sequência).
    ("log", Pipeline([("log1p", FunctionTransformer(np.log1p)),
                      ("z", StandardScaler())]), log_cols),
    # O resto (percentuais e notas de 1 a 5): só padroniza.
    ("z", StandardScaler(), demais_cols),
])
# fit_transform faz as duas coisas: aprende média e desvio de cada coluna e
# já aplica. O resultado sai na ordem em que os grupos foram declarados, por
# isso o [features] no fim recoloca tudo na ordem por bloco.
Z = pd.DataFrame(pre.fit_transform(modelagem[features]),
                 columns=log_cols + demais_cols)[features]
# Conferência: toda coluna tem que sair daqui com média 0 e desvio 1.
Z.describe().round(2).loc[["mean", "std"]]

## 3. Peso por bloco

Resta um desequilíbrio que a padronização não corrige. Após ela cada **coluna**
pesa igualmente, mas os blocos têm quantidades distintas de colunas: 9 de
criminalidade, 6 socioeconômicas e 7 de gestão. Como a distância soma a
contribuição de todas as colunas, o bloco mais numeroso pesa mais, e a sua
quantidade de colunas decorre do formato de divulgação da fonte, não da
importância do bloco.

A correção multiplica cada coluna por `1/√n`, sendo `n` o número de colunas do
bloco. O procedimento funciona porque as variâncias se somam: um bloco de `n`
colunas padronizadas contribui com `n`, e a divisão de cada coluna por `√n`
reduz essa contribuição a 1, seja o bloco de 6 ou de 9 colunas. Os três
passam a valer um terço cada.

A ideia deriva da Análise Fatorial Múltipla (Escofier e Pagès, 1994), método
criado para combinar grupos de variáveis de origens distintas sem permitir que
um grupo domine. A diferença deve ser registrada, pois o cálculo não é o
mesmo: a AFM divide cada grupo pela raiz do seu primeiro autovalor, de modo a
igualar o peso dos grupos **no primeiro eixo**, enquanto a divisão por `√n`
iguala a **variância total** de cada bloco. A versão adotada é mais simples de
explicar e de conferir, e entrega o objetivo pretendido: um terço da distância
para cada bloco, como mostra a figura abaixo.

In [ ]:
# Quantas colunas tem cada bloco (9, 6 e 7).
n_bloco = pd.Series([bloco_de[c] for c in features]).value_counts()
# Cada coluna recebe o peso do bloco a que pertence.
peso = pd.Series({c: 1 / np.sqrt(n_bloco[bloco_de[c]]) for c in features})
W = Z * peso        # W é a matriz final: padronizada e com peso aplicado
peso.groupby(bloco_de).first().round(3)

In [ ]:
def orcamento(M):
    """% da variância total que cada bloco tem."""
    # Com as colunas padronizadas, a variância de cada uma é o quanto ela
    # "ocupa" na distância. Somando por bloco dá para ver se os três estão
    # em pé de igualdade. ddof=0 usa a fórmula populacional, que é a mesma
    # que o StandardScaler usa.
    var = M.var(ddof=0)
    return (var.groupby(M.columns.map(bloco_de)).sum()
               .reindex(ORDEM_BLOCOS) / var.sum() * 100).round(1)

# Compara o antes (Z, só padronizada) com o depois (W, já com peso).
fig = plot_orcamento_blocos(orcamento(Z), orcamento(W))
salvar(fig, "figura_orcamento_blocos")

O gráfico compara as duas situações. Sem ponderação, a criminalidade
responde por 40,9% da distância, a gestão por 31,8% e o bloco socioeconômico
por 27,3%, diferença decorrente apenas da quantidade de colunas. Com a
ponderação, os três ficam em 33,3%, conforme previsto pela divisão por `√n`.

## 4. Figura 3 - dimensionalidade efetiva dos dados

O PCA (Análise de Componentes Principais) busca combinações das 22 colunas que
concentrem o máximo de variação. A primeira componente é a direção de maior
dispersão entre os municípios; a segunda é a direção de maior dispersão entre
as independentes da primeira, e assim sucessivamente.

O valor de interesse é **quantas componentes são necessárias para representar
80% da variação**. Um número baixo, como três ou quatro, indicaria que as 22
colunas repetem poucas ideias sob formas diferentes; um número alto indica que
cada bloco aporta informação própria.

Neste notebook o PCA é apenas diagnóstico: a clusterização opera sobre as 22
colunas ponderadas, não sobre as componentes.

In [ ]:
# PCA() sem n_components calcula todas as componentes, que é o que queremos
# para olhar a variância acumulada.
pca = PCA().fit(W)
fig = plot_variancia_pca(pca.explained_variance_ratio_)
salvar(fig, "figura3_variancia_pca")

São necessárias **12 das 22 componentes** para alcançar 80% da
variação, e a primeira explica isoladamente menos de um quarto. Duas leituras
decorrem disso:

1. **Os três blocos não medem o mesmo fenômeno.** É a mesma conclusão da
   matriz de Spearman do notebook 02, obtida por outro caminho: não há um
   conjunto reduzido de dimensões latentes que resuma a base.
2. **Alerta para a próxima entrega.** Dados dispersos em muitas dimensões
   constituem cenário desfavorável a métodos baseados em densidade, como o
   DBSCAN, pois as distâncias entre pontos tendem a se equalizar e o método
   classifica quase tudo como ruído. O registro é feito antes da execução, de
   modo a não configurar justificativa posterior.

## 5. Interpretação das duas primeiras componentes

Cada componente é uma soma ponderada das 22 colunas. Esses pesos, denominados
**cargas**, definem o que a componente representa: as colunas de carga alta,
positiva ou negativa, determinam a direção. Cargas de sinais opostos em uma
mesma componente indicam **contraste** entre os dois grupos de variáveis.

A tabela abaixo restringe-se às features com carga igual ou superior a 0,25 em
alguma das três primeiras componentes, de modo a manter a leitura enxuta.

In [ ]:
# components_ traz uma linha por componente; o .T vira para uma linha por
# feature, que é como a tabela fica legível.
cargas = pd.DataFrame(pca.components_[:3].T, index=features,
                      columns=["PC1", "PC2", "PC3"]).round(2)
# 0,25 é só um corte de leitura para a tabela caber; não muda nenhum cálculo.
fortes = cargas[(cargas.abs() >= 0.25).any(axis=1)].copy()
fortes["bloco"] = [bloco_de[c] for c in fortes.index]
# key=abs ordena pelo tamanho da carga, ignorando o sinal.
fortes.sort_values("PC1", key=abs, ascending=False)

In [ ]:
# transform() dá a posição de cada município nas componentes (os "escores").
escores = pca.transform(W)
# A urbanização não é eixo do gráfico, entra como cor: é um teste visual de
# que a PC1 é mesmo o eixo socioeconômico que as cargas sugerem.
fig = plot_pc1_pc2(escores, modelagem["taxa_urbanizacao"],
                   "taxa de urbanização (%)", pca.explained_variance_ratio_)
salvar(fig, "figura_pc1_pc2")

A **PC1 é um eixo de condição socioeconômica**: alfabetização,
urbanização, coleta de lixo, renda e esgoto apresentam as maiores cargas,
todas com o mesmo sinal. Daí o degradê organizado da esquerda para a direita
quando os pontos são coloridos pela urbanização, confirmação visual da leitura
das cargas. A **PC2** contrasta as notas de saúde e educação do IEGM (cargas
positivas) com as taxas de roubo (cargas negativas).

O aspecto mais relevante é que os municípios formam **uma nuvem contínua**,
sem grupos isolados. Não existem agrupamentos naturais à espera de descoberta:
os clusters da próxima entrega serão cortes dentro de um gradiente, e a
silhueta, medida de 0 a 1 do grau de separação entre grupos, tende a ser
baixa. Isso não invalida o resultado, mas delimita o que pode ser afirmado a
partir dele.

## 6. Matriz final para a clusterização

O `matriz_modelagem.csv` contém os 644 municípios com as 22 features
transformadas, padronizadas e ponderadas, acrescidas das duas colunas de
identificação. É o arquivo lido pela próxima entrega e a única saída deste
notebook além das figuras.

In [ ]:
saida = W.copy()
# Identificação na frente das features, para dar para conferir linha a linha
# depois sem precisar cruzar com outro arquivo.
saida.insert(0, "codigo_ibge", modelagem["codigo_ibge"])
saida.insert(1, "municipio", modelagem["municipio"])
saida.to_csv(DATA_PROCESSED / "matriz_modelagem.csv", index=False)
print(f"-> matriz_modelagem.csv: {saida.shape[0]} x {saida.shape[1]}")

In [ ]:
# Resumo das decisões desta fase, num formato que dá para conferir de relance
# e copiar para o artigo.
# cumsum() acumula a variância explicada; o >= 0.80 vira uma lista de
# verdadeiro/falso e o argmax devolve a posição do primeiro verdadeiro. O + 1
# é porque a contagem de posições começa em zero.
k80 = int(np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.80)) + 1
pd.DataFrame({
    "passo": ["subconjunto", "log1p", "padronização", "peso por bloco", "PCA"],
    "decisão": [
        f"{len(modelagem)} municípios (sem a capital)",
        f"{len(log_cols)} colunas: taxas criminais e valores em R$",
        "StandardScaler, ajustado neste subconjunto (ver apêndice 03b)",
        "1/√n: " + ", ".join(f"{b} {peso[[c for c in features if bloco_de[c] == b][0]]:.3f}"
                              for b in ORDEM_BLOCOS),
        f"{k80} componentes para 80% da variância",
    ],
})